# Face Mask Detection — EfficientNetB0 Transfer Learning

In Notebook 2, we used VGG16 for transfer learning. While it worked great (~96% accuracy), VGG16 is a massive model (138 million parameters, ~528MB).

Here, we will use **EfficientNetB0**, a modern, state-of-the-art model that is much smaller (~4 million parameters) and highly optimized for speed and efficiency. We'll freeze the pre-trained EfficientNetB0 base and train a simple classification head on top.

## 1. Imports

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import cv2

import tensorflow as tf
from keras.models import Sequential
from keras.layers import Dense, GlobalAveragePooling2D
from keras.applications.efficientnet import EfficientNetB0, preprocess_input
from keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

## 2. Load the Dataset

Same dataset folder structure and loading process as notebooks 1 and 2. We resize all images to 224x224.

In [ ]:
# Clone the dataset (run once if not already cloned)
# !git clone https://github.com/ricklon/pyimagesearch-face-mask-detector.git

dataset_path = "/content/pyimagesearch-face-mask-detector/dataset"
categories = ['without_mask', 'with_mask']   # 0 = no mask, 1 = mask

print("Classes:", os.listdir(dataset_path))

In [ ]:
data = []
for category in categories:
    path = os.path.join(dataset_path, category)
    label = categories.index(category)

    for file in os.listdir(path):
        img_path = os.path.join(path, file)
        img = cv2.imread(img_path)
        img = cv2.resize(img, (224, 224))
        data.append([img, label])

print(f"Total images loaded: {len(data)}")

## 3. Preprocess & Split

For EfficientNetB0, the model itself has a built-in rescaling layer () as part of its network definition. This means  is actually a pass-through function that does nothing. However, we'll still call it to follow standard keras application workflows.

In [ ]:
random.shuffle(data)

X = np.array([item[0] for item in data])
y = np.array([item[1] for item in data])

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Class distribution: no_mask={sum(y==0)}, mask={sum(y==1)}")

In [ ]:
# Apply the pass-through preprocessing
X = preprocess_input(X)

# 70/15/15 split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}")

## 4. Build the Model

Unlike VGG16 where we manually copied the dense layers, for EfficientNetB0 we will load the base model with  (dropping the final 1000-class classifier), freeze its weights, and add a  layer followed by a single  classification head.

This leaves us with only **1,281 trainable parameters** (vs 4,097 in VGG16) and a much smaller footprint!

In [ ]:
# Load pre-trained EfficientNetB0 base
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze the base model
base_model.trainable = False

# Build sequential model
model = Sequential()
model.add(base_model)
model.add(GlobalAveragePooling2D())
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

## 5. Train

We'll use similar callbacks and data augmentation as Notebook 2.

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=20,
    horizontal_flip=True,
    zoom_range=0.2,
    shear_range=0.2
)

callbacks = [
    EarlyStopping(patience=3, restore_best_weights=True),
    ModelCheckpoint('best_efficientnet_model.keras', save_best_only=True)
]

history = model.fit(
    datagen.flow(X_train, y_train, batch_size=32),
    epochs=30,
    validation_data=(X_val, y_val),
    callbacks=callbacks
)

## 6. Evaluate

In [ ]:
y_pred = (model.predict(X_test) > 0.5).astype(int)
print(classification_report(y_test, y_pred,
      target_names=['without_mask', 'with_mask']))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['accuracy'], label='train')
ax1.plot(history.history['val_accuracy'], label='val')
ax1.set_title('Accuracy')
ax1.legend()

ax2.plot(history.history['loss'], label='train')
ax2.plot(history.history['val_loss'], label='val')
ax2.set_title('Loss')
ax2.legend()

plt.tight_layout()
plt.show()

## 7. Takeaways

- EfficientNetB0 is a modern alternative that achieves similar (or better) accuracy to VGG16 but is **50x smaller** and much faster to run.
- Loading the base model with  lets us cleanly append a  layer and our  classifier, resulting in only **1,281 trainable parameters**.
- EfficientNet handles scaling and normalization internally, making it highly robust.
- **Verdict:** Transfer learning with EfficientNetB0 or MobileNetV2 is the optimal choice for real-time edge applications.